# Parse blastn-all results

 - export files that can be imported in Cytoscape

### Import libraries and set directory location

In [1]:
import sys
import pandas as pd
import numpy as np
import glob
from Bio.Blast import NCBIXML   #to parse XML files

## Functions

In [22]:
# Dada2 Banzai Output Functions
levels = ['Kingdom', 'Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species']

def make_metadata(infile):
    df = pd.read_csv(infile,index_col=0)
    df.index.names = ['sample_name']
    #df['date'] = pd.to_datetime(df['date'])
    print('Number samples:', len(df.index))
    return df

def make_taxa_asv(infile):
    df = pd.read_csv(infile)
    #df = df.drop('Unnamed: 0', axis=0)
    #df = df.rename(columns= {'Unnamed: 0':'ASV'})
    #df.set_index('ASV', inplace=True)
    print('Number ASVs:', len(df.index))
    return df

#from metadata file, limit OTU table and taxa table to those present in those samples
def from_metadata_to_taxareads(meta_data, otu_table, taxa_table):
    #standard M6 output; sample_names as index; OTUs as index
    cols = list(meta_data)
    otu_lim = pd.concat([meta_data, otu_table.T],join='inner', axis=1)
    otu_lim.drop(cols, inplace=True, axis=1)
    otu_lim=otu_lim.T
    otu_lim['Total']=otu_lim.sum(axis=1)
    otu_lim = otu_lim.loc[otu_lim['Total']>0]
    otu_lim.drop('Total', axis=1, inplace=True)
    cols=list(otu_lim)
    taxa_lim=pd.concat([otu_lim, taxa_table], axis=1, join='inner')
    taxa_lim.drop(cols, inplace=True, axis=1)
    return otu_lim, taxa_lim
    
#From fasta file create pandas df of ASV and sequence
def from_fasta_to_df(file):
    print(file)
    with open(file) as f:
        Ids=[]
        seqs =[]
        for strline in f:
            if strline[0]=='>':
                Ids.append(strline[1:].strip())
            else:
                seqs.append(strline.strip())
    print('Number of Ids:',len(Ids))
    print('Number of Seqs:',len(seqs))
    seq_dict = dict(zip(Ids, seqs))
    #make pandas df
    df= pd.DataFrame.from_dict(seq_dict,orient='index', columns=['sequence'])
    return df

In [4]:
directory = '/Users/kpitz/github/GLOMICON/intercomparison/Merged_Datasets/'
prefix = 'GLOMICON'
marker = '18S'
blast_file = '/Users/kpitz/github/GLOMICON/intercomparison/Merged_Datasets/blastn_all/GLOMICON_blastn_all_102025.xml'
print(blast_file)

'''taxa_table = '/Users/kpitz/github/GLOMICON/intercomparison/Merged_Datasets/data/merged_original_taxonomy/GLOMICON_taxa_blastnr.csv'
print(taxa_table)
seq_table = '/Users/kpitz/github/GLOMICON/intercomparison/Merged_Datasets/data/merged_original_taxonomy/GLOMICON_seq_merged.csv'
print(seq_table)'''

/Users/kpitz/github/GLOMICON/intercomparison/Merged_Datasets/blastn_all/GLOMICON_blastn_all_102025.xml


"taxa_table = '/Users/kpitz/github/GLOMICON/intercomparison/Merged_Datasets/data/merged_original_taxonomy/GLOMICON_taxa_blastnr.csv'\nprint(taxa_table)\nseq_table = '/Users/kpitz/github/GLOMICON/intercomparison/Merged_Datasets/data/merged_original_taxonomy/GLOMICON_seq_merged.csv'\nprint(seq_table)"

In [23]:
# fasta file used in blast analysis:
file = '../PR2/GLOMICON_seq_merged_unique.fasta'
df = from_fasta_to_df(file)
uniq_seqs = df.copy()
uniq_seqs

../PR2/GLOMICON_seq_merged_unique.fasta
Number of Ids: 13542
Number of Seqs: 13542


,sequence
ASV_1,CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGG...
ASV_2,GCACCTACCGATTGAATGGTCCGGTGAAGACTCGGGATTGTGGTCT...
ASV_3,GCACCTACCGATTGAATGGTCCGGTGAGGCCTCGGGATCGTGGCGA...
ASV_4,GCACCTACCGATTGAATGGTCCGGTGAAGCCTCGGGATTGTGGTTG...
ASV_5,GCTCCTACCGATTGAGTGATCCGGTGAATAATTCGGACTGCAGCAG...
...,...
ASV_14543,TAGCGTATATTAATGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATT...
ASV_14544,TAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGAATT...
ASV_14545,TAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATT...
ASV_14546,TAGCGTATACTAATGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATT...


In [24]:
# csv file of every sequence in project - ASV IDs may shift if upstream processing is rerun
file = '../data/merged_original_taxonomy/GLOMICON_seq_merged.csv'
df = pd.read_csv(file)
df.set_index('ASV', inplace=True)
seq_all = df.copy()
seq_all

,sequence,Analyzing_Institute
ASV,,
ASV_1,CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGG...,UDAL
ASV_2,GCACCTACCGATTGAATGGTCCGGTGAAGACTCGGGATTGTGGTCT...,MBARI
ASV_3,GCACCTACCGATTGAATGGTCCGGTGAGGCCTCGGGATCGTGGCGA...,MBARI
ASV_4,GCACCTACCGATTGAATGGTCCGGTGAAGCCTCGGGATTGTGGTTG...,MBARI
ASV_5,GCTCCTACCGATTGAGTGATCCGGTGAATAATTCGGACTGCAGCAG...,MBARI
...,...,...
ASV_14543,TAGCGTATATTAATGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATT...,AWI
ASV_14544,TAGCGTATACTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGAATT...,AWI
ASV_14545,TAGCGTATACTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGAATT...,AWI


In [61]:
# asv file
file = '../data/merged_original_taxonomy/GLOMICON_asv_merged.csv'
df = pd.read_csv(file)
df.set_index('ASV', inplace=True)
asv_all = df.copy()
asv_all

,Davenport_02_0008,Davenport_06_0008,Davenport_11_0008,Davenport_15_0008,Davenport_19_0008,Framstrait_01_0008,Framstrait_05_0008,Framstrait_09_0008,Framstrait_13_0008,Framstrait_17_0008,...,E-G30-eMockB-18,E-G29-eMockA-11,E-G32-bMock2,G11r-DAL30,E-G35-bMock22,G20r-ROS12,E-G31-eMockB-20,E-G34-bMock20,E-G1-AWI11,E-G33-bMock9
ASV,,,,,,,,,,,,,,,,,,,,,
ASV_1,0,0,0,0,0,0,0,0,0,0,...,1712.0,1266.0,33675.0,124.0,34286.0,4.0,1383.0,56835.0,360.0,34134.0
ASV_2,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ASV_3,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ASV_4,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ASV_5,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ASV_14543,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ASV_14544,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ASV_14545,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [59]:
# metadata file
file = '../data/merged_original_taxonomy/GLOMICON_meta_merged.csv'
df = pd.read_csv(file)
df.set_index('sample_name', inplace=True)
meta_all = df.copy()
meta_all

,Analyzing_Institute,Collecting_Institute,site
sample_name,,,
Davenport_02_0008,AWI,MBARI,Monterey Bay
Davenport_06_0008,AWI,MBARI,Monterey Bay
Davenport_11_0008,AWI,MBARI,Monterey Bay
Davenport_15_0008,AWI,MBARI,Monterey Bay
Davenport_19_0008,AWI,MBARI,Monterey Bay
...,...,...,...
G20r-ROS12,UDAL,SBR,Roscoff
E-G31-eMockB-20,UDAL,EVENMOCK,EVENMOCK
E-G34-bMock20,UDAL,BLOOMMOCK,BLOOMMOCK


## Import Data

In [11]:
'''#sequence table
df = pd.read_csv(seq_table)
df = df.rename(columns={'Unnamed: 0':'ASV'})
df.set_index('ASV', inplace=True)
seq = df.copy()
seq.head()

# Dada2, MEGAN, Blastn results
df = pd.read_csv(taxa_table)
df = df.rename(columns={'Unnamed: 0':'ASV'})
df.set_index('ASV', inplace=True)
taxa = df.copy()
taxa.head()
df'''

,Domain,Kingdom,Phylum,Class,Order,Family,Genus,Species
ASV,,,,,,,,
ASV_1950,Archaea,NaN,unknown,unknown,unknown,unknown,unknown,archaeon
ASV_9581,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ASV_63,Archaea,NaN,unknown,unknown,unknown,unknown,unknown,archaeon
ASV_3182,Archaea,NaN,unknown,unknown,unknown,unknown,unknown,archaeon
ASV_603,Archaea,NaN,unknown,unknown,unknown,unknown,unknown,archaeon
...,...,...,...,...,...,...,...,...
ASV_12927,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ASV_12880,Eukaryota,NaN,Bacillariophyta,Bacillariophyceae,Mastogloiales,Achnanthaceae,Achnanthes,NaN
ASV_14103,Eukaryota,NaN,unknown,unknown,unknown,unknown,unknown,uncultured marine eukaryote


## Parse XML file to get top hits per ASV and percent ID and bitscore values

In [6]:
# Only save top 2% of bitscore hits to query
print('Blast XML file: ', blast_file)
hit_dict ={}  #Dictionary to store hits
result_handle =open(blast_file)
#parse blast records
blast_records= NCBIXML.parse(result_handle)
for blast_record in blast_records:
    query = blast_record.query
    hit_ID=0 #hit counter
    for alignment in blast_record.alignments:
        hit_ID+=1
        key = (query, hit_ID)
        hsp = alignment.hsps[0] #only look at top hsp per alignment to genbank sequence
        #set limits on evalue and bitscore and % Identity
        per_iden = hsp.identities/float(hsp.align_length)
        per_iden = (per_iden *100)
        per_iden = round(per_iden, 2)
        evalue = hsp.expect
        bitscore = int(hsp.score)
        # set bitscore limit for hits, within top 2%
        if hit_ID ==1:
            top_bitscore = bitscore
            min_bitscore = top_bitscore - (top_bitscore*.02)
        if bitscore < min_bitscore:
            #continue to next alignment if bitscore too low
            continue
        #alignment length
        align_len = hsp.align_length
        # save results to dictionary
        value = (evalue, bitscore, per_iden, alignment.hit_def, align_len)
        hit_dict[key]=value
print ('Done parsing ', blast_file)

Blast XML file:  /Users/kpitz/github/GLOMICON/intercomparison/Merged_Datasets/blastn_all/GLOMICON_blastn_all_102025.xml
Done parsing  /Users/kpitz/github/GLOMICON/intercomparison/Merged_Datasets/blastn_all/GLOMICON_blastn_all_102025.xml


In [7]:
#Save Blast Results to file
df = pd.DataFrame(hit_dict)
df=df.T
df.columns = ['eval','bitscore', '%ID', 'hit_def', 'align_len']
df.reset_index(inplace=True)
df = df.rename(columns={'level_0': 'ASV', 'level_1': 'Hit_number'})

df

ASV_17


,ASV,Hit_number,eval,bitscore,%ID,hit_def,align_len
0,ASV_1,1,0.0,350,100.0,ASV_17,350
1,ASV_1,2,0.0,350,100.0,ASV_1,350
2,ASV_1,3,0.0,347,100.0,ASV_13069,347
3,ASV_1,4,0.0,347,100.0,ASV_12196,347
4,ASV_1,5,0.0,347,100.0,ASV_10974,347
...,...,...,...,...,...,...,...
21542,ASV_14547,1,0.0,369,100.0,ASV_14547,369
21543,ASV_14547,2,0.0,367,100.0,ASV_11970,367
21544,ASV_14547,3,0.0,366,99.73,ASV_11336,369
21545,ASV_14547,4,0.0,364,99.73,ASV_12928,367


In [19]:
#Save Blast Results to file
df = pd.DataFrame(hit_dict)
df=df.T
df.columns = ['eval','bitscore', '%ID', 'hit_def', 'align_len']
df.reset_index(inplace=True)
df = df.rename(columns={'level_0': 'ASV', 'level_1': 'Hit_number'})
#df.set_index('ASV', inplace=True)
#df = pd.concat([df, uniq_seqs], axis=1 )
df = df.join(uniq_seqs, on='ASV')
df = df.rename(columns={'sequence':'sequence_query'})
df = df.join(uniq_seqs, on='hit_def')
df = df.rename(columns={'sequence':'sequence_hit'})
df = df.drop(['ASV', 'hit_def'], axis=1)
df.set_index('sequence_query', inplace=True)
all_hits = df.copy()
# save results to file:
file = '/Users/kpitz/github/GLOMICON/intercomparison/Merged_Datasets/blastn_all/blastn_all_hits.csv'
df.to_csv(file)
df

,Hit_number,eval,bitscore,%ID,align_len,sequence_hit
sequence_query,,,,,,
CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATTTCGGGTCGGGCCGAGCGGTCTGCCGATGGGTATGCACTGTTTGGCGCGGCCTTCTTTCCGGAGACCGCGGCTACTCTTAACTGAGCGGGCGTGGGAGACGGATCGTTTACTTTGAAAAAATCAGAGTGTTTCTAGCAGGCAGCTCGCTCTTGCATAGGTTAGCATGGGATAATTTAATAGGACTCTGGTGCTATTTTGTTGGTTTCGAACACCGGAGTAATGATTAAAAGGGGCAGTCAGGGGCACTCGTATTCCGTCGAGAGAGGTGAAATTCTCAGACCAATGGAAGACGAACCACTGC,1,0.0,350,100.0,350,AGCTCCAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTA...
CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATTTCGGGTCGGGCCGAGCGGTCTGCCGATGGGTATGCACTGTTTGGCGCGGCCTTCTTTCCGGAGACCGCGGCTACTCTTAACTGAGCGGGCGTGGGAGACGGATCGTTTACTTTGAAAAAATCAGAGTGTTTCTAGCAGGCAGCTCGCTCTTGCATAGGTTAGCATGGGATAATTTAATAGGACTCTGGTGCTATTTTGTTGGTTTCGAACACCGGAGTAATGATTAAAAGGGGCAGTCAGGGGCACTCGTATTCCGTCGAGAGAGGTGAAATTCTCAGACCAATGGAAGACGAACCACTGC,2,0.0,350,100.0,350,CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGG...
CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATTTCGGGTCGGGCCGAGCGGTCTGCCGATGGGTATGCACTGTTTGGCGCGGCCTTCTTTCCGGAGACCGCGGCTACTCTTAACTGAGCGGGCGTGGGAGACGGATCGTTTACTTTGAAAAAATCAGAGTGTTTCTAGCAGGCAGCTCGCTCTTGCATAGGTTAGCATGGGATAATTTAATAGGACTCTGGTGCTATTTTGTTGGTTTCGAACACCGGAGTAATGATTAAAAGGGGCAGTCAGGGGCACTCGTATTCCGTCGAGAGAGGTGAAATTCTCAGACCAATGGAAGACGAACCACTGC,3,0.0,347,100.0,347,TAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATT...
CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATTTCGGGTCGGGCCGAGCGGTCTGCCGATGGGTATGCACTGTTTGGCGCGGCCTTCTTTCCGGAGACCGCGGCTACTCTTAACTGAGCGGGCGTGGGAGACGGATCGTTTACTTTGAAAAAATCAGAGTGTTTCTAGCAGGCAGCTCGCTCTTGCATAGGTTAGCATGGGATAATTTAATAGGACTCTGGTGCTATTTTGTTGGTTTCGAACACCGGAGTAATGATTAAAAGGGGCAGTCAGGGGCACTCGTATTCCGTCGAGAGAGGTGAAATTCTCAGACCAATGGAAGACGAACCACTGC,4,0.0,347,100.0,347,TAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATT...
CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATTTCGGGTCGGGCCGAGCGGTCTGCCGATGGGTATGCACTGTTTGGCGCGGCCTTCTTTCCGGAGACCGCGGCTACTCTTAACTGAGCGGGCGTGGGAGACGGATCGTTTACTTTGAAAAAATCAGAGTGTTTCTAGCAGGCAGCTCGCTCTTGCATAGGTTAGCATGGGATAATTTAATAGGACTCTGGTGCTATTTTGTTGGTTTCGAACACCGGAGTAATGATTAAAAGGGGCAGTCAGGGGCACTCGTATTCCGTCGAGAGAGGTGAAATTCTCAGACCAATGGAAGACGAACCACTGC,5,0.0,347,100.0,347,TAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATT...
...,...,...,...,...,...,...
TAGCGTATATTTAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATTTCGGTTGAGAACGGCCGGTCCGCCGTTTGGTGTGCACTGGCTGGTCTCAACTTCCTGTAGAGGACGCGCTCTGGGTTAACGCTCGGACGCGGAGTCTACGTGGTTACTTTGAAAAAATTAGAGTGTTCAAAGCGGGCTTACGCTTGAATATTTCAGCATGGAATAACACTATAGGACTCCTGTCCTATTTCGTTGGTCTCGGGACGGGAGTAATGATTAAGAGGAACAGTTGGGGGCATTCGTATTTCATTGTCAGAGGTGAAATTCTTGGATTTATGAAAGACGAACTTCTGCGAAAGCATTTGCCAAGGATGTTTTCATTA,1,0.0,369,100.0,369,TAGCGTATATTTAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATT...
TAGCGTATATTTAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATTTCGGTTGAGAACGGCCGGTCCGCCGTTTGGTGTGCACTGGCTGGTCTCAACTTCCTGTAGAGGACGCGCTCTGGGTTAACGCTCGGACGCGGAGTCTACGTGGTTACTTTGAAAAAATTAGAGTGTTCAAAGCGGGCTTACGCTTGAATATTTCAGCATGGAATAACACTATAGGACTCCTGTCCTATTTCGTTGGTCTCGGGACGGGAGTAATGATTAAGAGGAACAGTTGGGGGCATTCGTATTTCATTGTCAGAGGTGAAATTCTTGGATTTATGAAAGACGAACTTCTGCGAAAGCATTTGCCAAGGATGTTTTCATTA,2,0.0,367,100.0,367,TAGCGTATATTTAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATT...
TAGCGTATATTTAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATTTCGGTTGAGAACGGCCGGTCCGCCGTTTGGTGTGCACTGGCTGGTCTCAACTTCCTGTAGAGGACGCGCTCTGGGTTAACGCTCGGACGCGGAGTCTACGTGGTTACTTTGAAAAAATTAGAGTGTTCAAAGCGGGCTTACGCTTGAATATTTCAGCATGGAATAACACTATAGGACTCCTGTCCTATTTCGTTGGTCTCGGGACGGGAGTAATGATTAAGAGGAACAGTTGGGGGCATTCGTATTTCATTGTCAGAGGTGAAATTCTTGGATTTATGAAAGACGAACTTCTGCGAAAGCATTTGCCAAGGATGTTTTCATTA,3,0.0,366,99.73,369,TAGCGTATATTTAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATT...


## Handle conversion to ASV IDs and add institute identities

In [26]:
seq_all

,sequence,Analyzing_Institute
ASV,,
ASV_1,CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGG...,UDAL
ASV_2,GCACCTACCGATTGAATGGTCCGGTGAAGACTCGGGATTGTGGTCT...,MBARI
ASV_3,GCACCTACCGATTGAATGGTCCGGTGAGGCCTCGGGATCGTGGCGA...,MBARI
ASV_4,GCACCTACCGATTGAATGGTCCGGTGAAGCCTCGGGATTGTGGTTG...,MBARI
ASV_5,GCTCCTACCGATTGAGTGATCCGGTGAATAATTCGGACTGCAGCAG...,MBARI
...,...,...
ASV_14543,TAGCGTATATTAATGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATT...,AWI
ASV_14544,TAGCGTATACTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGAATT...,AWI
ASV_14545,TAGCGTATACTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGAATT...,AWI


In [32]:
df = all_hits.copy()
df = df.reset_index()
df = df.drop_duplicates('sequence_query')
df

,sequence_query,Hit_number,eval,bitscore,%ID,align_len,sequence_hit
0,CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGG...,1,0.0,350,100.0,350,AGCTCCAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTA...
11,GCACCTACCGATTGAATGGTCCGGTGAAGACTCGGGATTGTGGTCT...,1,0.0,127,100.0,127,GCACCTACCGATTGAATGGTCCGGTGAAGACTCGGGATTGTGGTCT...
13,GCACCTACCGATTGAATGGTCCGGTGAGGCCTCGGGATCGTGGCGA...,1,0.0,127,100.0,127,GCACCTACCGATTGAATGGTCCGGTGAGGCCTCGGGATCGTGGCGA...
18,GCACCTACCGATTGAATGGTCCGGTGAAGCCTCGGGATTGTGGTTG...,1,0.0,127,100.0,127,GCACCTACCGATTGAATGGTCCGGTGAAGCCTCGGGATTGTGGTTG...
20,GCTCCTACCGATTGAGTGATCCGGTGAATAATTCGGACTGCAGCAG...,1,0.0,126,100.0,126,GCTCCTACCGATTGAGTGATCCGGTGAATAATTCGGACTGCAGCAG...
...,...,...,...,...,...,...,...
21537,TAGCGTATATTAATGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATT...,1,0.0,381,100.0,381,TAGCGTATATTAATGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATT...
21538,TAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGAATT...,1,0.0,364,100.0,364,TAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGAATT...
21539,TAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATT...,1,0.0,368,100.0,368,TAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATT...
21540,TAGCGTATACTAATGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATT...,1,0.0,365,100.0,365,TAGCGTATACTAATGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATT...


In [43]:
# indexed by sequence:
df = seq_all.copy()
df = df.reset_index().set_index('sequence')
df  = df.join(all_hits, lsuffix = '_query')
#df  = df.join(all_hits, on='sequence_hit', rsuffix = '_hit')
df  = df.join(seq_all.reset_index().set_index('sequence'), on='sequence_hit', rsuffix = '_hit')
#df = all_hits.copy()

# save results
file = '/Users/kpitz/github/GLOMICON/intercomparison/Merged_Datasets/blastn_all/blastn_all_sequence_hits_withdups.csv'
df.to_csv(file)

blast_results_byseq = df.copy()
df

,ASV,Analyzing_Institute,Hit_number,eval,bitscore,%ID,align_len,sequence_hit,ASV_hit,Analyzing_Institute_hit
sequence,,,,,,,,,,
CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATTTCGGGTCGGGCCGAGCGGTCTGCCGATGGGTATGCACTGTTTGGCGCGGCCTTCTTTCCGGAGACCGCGGCTACTCTTAACTGAGCGGGCGTGGGAGACGGATCGTTTACTTTGAAAAAATCAGAGTGTTTCTAGCAGGCAGCTCGCTCTTGCATAGGTTAGCATGGGATAATTTAATAGGACTCTGGTGCTATTTTGTTGGTTTCGAACACCGGAGTAATGATTAAAAGGGGCAGTCAGGGGCACTCGTATTCCGTCGAGAGAGGTGAAATTCTCAGACCAATGGAAGACGAACCACTGC,ASV_1,UDAL,1,0.0,350,100.0,350,AGCTCCAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTA...,ASV_17,SBR
CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATTTCGGGTCGGGCCGAGCGGTCTGCCGATGGGTATGCACTGTTTGGCGCGGCCTTCTTTCCGGAGACCGCGGCTACTCTTAACTGAGCGGGCGTGGGAGACGGATCGTTTACTTTGAAAAAATCAGAGTGTTTCTAGCAGGCAGCTCGCTCTTGCATAGGTTAGCATGGGATAATTTAATAGGACTCTGGTGCTATTTTGTTGGTTTCGAACACCGGAGTAATGATTAAAAGGGGCAGTCAGGGGCACTCGTATTCCGTCGAGAGAGGTGAAATTCTCAGACCAATGGAAGACGAACCACTGC,ASV_1,UDAL,2,0.0,350,100.0,350,CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGG...,ASV_1,UDAL
CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATTTCGGGTCGGGCCGAGCGGTCTGCCGATGGGTATGCACTGTTTGGCGCGGCCTTCTTTCCGGAGACCGCGGCTACTCTTAACTGAGCGGGCGTGGGAGACGGATCGTTTACTTTGAAAAAATCAGAGTGTTTCTAGCAGGCAGCTCGCTCTTGCATAGGTTAGCATGGGATAATTTAATAGGACTCTGGTGCTATTTTGTTGGTTTCGAACACCGGAGTAATGATTAAAAGGGGCAGTCAGGGGCACTCGTATTCCGTCGAGAGAGGTGAAATTCTCAGACCAATGGAAGACGAACCACTGC,ASV_1,UDAL,3,0.0,347,100.0,347,TAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATT...,ASV_13102,AWI
CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATTTCGGGTCGGGCCGAGCGGTCTGCCGATGGGTATGCACTGTTTGGCGCGGCCTTCTTTCCGGAGACCGCGGCTACTCTTAACTGAGCGGGCGTGGGAGACGGATCGTTTACTTTGAAAAAATCAGAGTGTTTCTAGCAGGCAGCTCGCTCTTGCATAGGTTAGCATGGGATAATTTAATAGGACTCTGGTGCTATTTTGTTGGTTTCGAACACCGGAGTAATGATTAAAAGGGGCAGTCAGGGGCACTCGTATTCCGTCGAGAGAGGTGAAATTCTCAGACCAATGGAAGACGAACCACTGC,ASV_1,UDAL,4,0.0,347,100.0,347,TAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATT...,ASV_12283,AWI
CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATTTCGGGTCGGGCCGAGCGGTCTGCCGATGGGTATGCACTGTTTGGCGCGGCCTTCTTTCCGGAGACCGCGGCTACTCTTAACTGAGCGGGCGTGGGAGACGGATCGTTTACTTTGAAAAAATCAGAGTGTTTCTAGCAGGCAGCTCGCTCTTGCATAGGTTAGCATGGGATAATTTAATAGGACTCTGGTGCTATTTTGTTGGTTTCGAACACCGGAGTAATGATTAAAAGGGGCAGTCAGGGGCACTCGTATTCCGTCGAGAGAGGTGAAATTCTCAGACCAATGGAAGACGAACCACTGC,ASV_1,UDAL,5,0.0,347,100.0,347,TAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATT...,ASV_10403,AWI
...,...,...,...,...,...,...,...,...,...,...
TAGCGTATATTTAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATTTCGGTTGAGAACGGCCGGTCCGCCGTTTGGTGTGCACTGGCTGGTCTCAACTTCCTGTAGAGGACGCGCTCTGGGTTAACGCTCGGACGCGGAGTCTACGTGGTTACTTTGAAAAAATTAGAGTGTTCAAAGCGGGCTTACGCTTGAATATTTCAGCATGGAATAACACTATAGGACTCCTGTCCTATTTCGTTGGTCTCGGGACGGGAGTAATGATTAAGAGGAACAGTTGGGGGCATTCGTATTTCATTGTCAGAGGTGAAATTCTTGGATTTATGAAAGACGAACTTCTGCGAAAGCATTTGCCAAGGATGTTTTCATTA,ASV_14547,AWI,1,0.0,369,100.0,369,TAGCGTATATTTAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATT...,ASV_14547,AWI
TAGCGTATATTTAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATTTCGGTTGAGAACGGCCGGTCCGCCGTTTGGTGTGCACTGGCTGGTCTCAACTTCCTGTAGAGGACGCGCTCTGGGTTAACGCTCGGACGCGGAGTCTACGTGGTTACTTTGAAAAAATTAGAGTGTTCAAAGCGGGCTTACGCTTGAATATTTCAGCATGGAATAACACTATAGGACTCCTGTCCTATTTCGTTGGTCTCGGGACGGGAGTAATGATTAAGAGGAACAGTTGGGGGCATTCGTATTTCATTGTCAGAGGTGAAATTCTTGGATTTATGAAAGACGAACTTCTGCGAAAGCATTTGCCAAGGATGTTTTCATTA,ASV_14547,AWI,2,0.0,367,100.0,367,TAGCGTATATTTAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATT...,ASV_11992,AWI
TAGCGTATATTTAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATTTCGGTTGAGAACGGCCGGTCCGCCGTTTGGTGTGCACTGGCTGGTCTCAACTTCCTGTAGAGGACGCGCTCTGGGTTAACGCTCGGACGCGGAGTCTACGTGGTTACTTTGAAAAAATTAGAGTGTTCAAAGCGGGCTTACGCTTGAATATTTCAGCATGGAATAACACTATAGGACTCCTGTCCTATTTCGTTGGTCTCGGGACGGGAGTAATGATTAAGAGGAACAGTTGGGGGCATTCGTATTTCATTGTCAGAGGTGAAATTCTTGGATTTATGAAAGACGAACTTCTGCGAAAGCATTTGCCAAGGATGTTTTCATTA,ASV_14547,AWI,3,0.0,366,99.73,369,TAGCGTATATTTAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATT...,ASV_11315,AWI


## Merge with blastn results against PR2 (top hits)

In [45]:
seq_all

,sequence,Analyzing_Institute
ASV,,
ASV_1,CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGG...,UDAL
ASV_2,GCACCTACCGATTGAATGGTCCGGTGAAGACTCGGGATTGTGGTCT...,MBARI
ASV_3,GCACCTACCGATTGAATGGTCCGGTGAGGCCTCGGGATCGTGGCGA...,MBARI
ASV_4,GCACCTACCGATTGAATGGTCCGGTGAAGCCTCGGGATTGTGGTTG...,MBARI
ASV_5,GCTCCTACCGATTGAGTGATCCGGTGAATAATTCGGACTGCAGCAG...,MBARI
...,...,...
ASV_14543,TAGCGTATATTAATGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATT...,AWI
ASV_14544,TAGCGTATACTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGAATT...,AWI
ASV_14545,TAGCGTATACTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGAATT...,AWI


In [49]:
file = '/Users/kpitz/github/GLOMICON/intercomparison/Merged_Datasets/PR2/PR2_blast_results_tophits.csv'
df = pd.read_csv(file)
df = df.join(uniq_seqs, on='ASV')
df.set_index('sequence', inplace=True)
df = df.drop('ASV', axis=1)
df = seq_all.reset_index().set_index('sequence').join(df)
#export just PR2 blast results:
file = '/Users/kpitz/github/GLOMICON/intercomparison/Merged_Datasets/network/blastn_PR2_sequence_hits_withdups.csv'
df.to_csv(file)

pr2_blast_results = df.copy()
print(list(pr2_blast_results))
df

['ASV', 'Analyzing_Institute', 'Hit_number', 'eval', 'bitscore', '%ID', 'hit_def', 'align_len', 'PR2_ID', 'PR2_marker', 'location', 'nuc', 'Domain', 'Supergroup', 'Division', 'Subdivision', 'Class', 'Order', 'Family', 'Genus', 'Species']


,ASV,Analyzing_Institute,Hit_number,eval,bitscore,%ID,hit_def,align_len,PR2_ID,PR2_marker,...,nuc,Domain,Supergroup,Division,Subdivision,Class,Order,Family,Genus,Species
sequence,,,,,,,,,,,,,,,,,,,,,
CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATTTCGGGTCGGGCCGAGCGGTCTGCCGATGGGTATGCACTGTTTGGCGCGGCCTTCTTTCCGGAGACCGCGGCTACTCTTAACTGAGCGGGCGTGGGAGACGGATCGTTTACTTTGAAAAAATCAGAGTGTTTCTAGCAGGCAGCTCGCTCTTGCATAGGTTAGCATGGGATAATTTAATAGGACTCTGGTGCTATTTTGTTGGTTTCGAACACCGGAGTAATGATTAAAAGGGGCAGTCAGGGGCACTCGTATTCCGTCGAGAGAGGTGAAATTCTCAGACCAATGGAAGACGAACCACTGC,ASV_1,UDAL,1.0,0.000000e+00,350.0,100.00,KC488454.1.1667_U|18S_rRNA|nucleus|clone_HL5aS...,350.0,KC488454.1.1667_U,18S_rRNA,...,clone_HL5aSF04.03,Eukaryota,Haptista,Haptophyta,Haptophyta_X,Prymnesiophyceae,Phaeocystales,Phaeocystaceae,Phaeocystis,Phaeocystis_pouchetii
GCACCTACCGATTGAATGGTCCGGTGAAGACTCGGGATTGTGGTCTGGCTCCTTCATTGGGGCCAGACCGTGAGAACTTGTCTGAACCTTATCATTTAGAGGAAGGTGAAGTCGTAACAAGGTTTCC,ASV_2,MBARI,1.0,3.815610e-61,127.0,100.00,AJ535170.1.1763_U|18S_rRNA|nucleus|clone_p928|...,127.0,AJ535170.1.1763_U,18S_rRNA,...,clone_p928,Eukaryota,TSAR,Stramenopiles,Gyrista,Mediophyceae,Thalassiosirales,Thalassiosiraceae,Conticribra,Conticribra_weissflogii
GCACCTACCGATTGAATGGTCCGGTGAGGCCTCGGGATCGTGGCGAACTTTCTTCATTGGAGGTGAGCTGTGAGAACTTGTCCAAATCTTATCATTTAGAGGAAGGTGAAGTCGTAACAAGGTTTCC,ASV_3,MBARI,1.0,3.842700e-56,118.0,97.64,KJ961704.1.1788_U|18S_rRNA|nucleus|strain_52|E...,127.0,KJ961704.1.1788_U,18S_rRNA,...,strain_52,Eukaryota,TSAR,Stramenopiles,Gyrista,Coscinodiscophyceae,Melosirales,Melosiraceae,Melosira,Melosira_sp.
GCACCTACCGATTGAATGGTCCGGTGAAGCCTCGGGATTGTGGTTGGTTTCCTTTATTGGAATCTGACCACGAGAACCTGTCTAAACCTTATCATTTAGAGGAAGGTGAAGTCGTAACAAGGTTTCC,ASV_4,MBARI,1.0,3.815610e-61,127.0,100.00,GU373969.1.1789_U|18S_rRNA|nucleus|strain_CCMP...,127.0,GU373969.1.1789_U,18S_rRNA,...,strain_CCMP1440,Eukaryota,TSAR,Stramenopiles,Gyrista,Bacillariophyceae,Bacillariales,Bacillariaceae,Pseudo-nitzschia,Pseudo-nitzschia_seriata
GCTCCTACCGATTGAGTGATCCGGTGAATAATTCGGACTGCAGCAGTGTTCAGTTCCTGAACGTTGCAGTGGAAAGTTTAGTGAACCTTATCACTTAGAGGAAGGAGAAGTCGTAACAAGGTTTCC,ASV_5,MBARI,1.0,1.358880e-60,126.0,100.00,MN965778.1.1765_U|18S_rRNA|nucleus||Eukaryota|...,126.0,MN965778.1.1765_U,18S_rRNA,...,NaN,Eukaryota,TSAR,Alveolata,Dinoflagellata,Dinophyceae,Gymnodiniales,Kareniaceae,Shimiella,Shimiella_gracilenta
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TAGCGTATATTAATGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATTTCTACCTATTTTGCCCGTTCCGGATGATATTCATCGTTCGGTAATGGAGCTGAAGTGGGTATCCTATCATGAAGTTGCGGCCTCGTGTCGTCGCTGAGTGGTTCTTTTACTTTGAAAAAATTAGAGTGTTTAAAGCAGGCTTGCGCAAGAATACATTAGCATGGAATAATAACATAGGACTTTGGTCTCTATTTTGTTGGTTTCTAGGACTAAAGTAATGATTGATAGGGATAGTTGGGGGTGCTAGTATTGAGCGGCCAGAGGTGAAATTCTTAGATTCGCTCAAGACTAACTTATGCGAAAGCATTCACCAAGGATGTCTTCTTTA,ASV_14543,AWI,1.0,0.000000e+00,371.0,99.73,FR874369.1.1800_U|18S_rRNA|nucleus|clone_1802B...,374.0,FR874369.1.1800_U,18S_rRNA,...,clone_1802B12,Eukaryota,TSAR,Rhizaria,Cercozoa,Filosa-Imbricatea,Filosa-Imbricatea_X,Novel-clade-2,Novel-clade-2_X,Novel-clade-2_X_sp.
TAGCGTATACTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGAATTTCTGGATGTGAAAGGTGGTCCGTTCCTCACGGAATGCGTACTTTCCATTTATGTCCATCCTCGGAGTTAGTGTGTATTACTTTAATCGGTCGTGCATACAACCTCCGTCATTTACTGTGAGCAAAATAGAGTGTTCAAAGCAGGCTTATGCCGTTGAATATATTAGCATGGAATAATAAGATATGACTTTGGTCTATTTTGTTGGTTTGCACTCCAAAGTAATGATTAATAGGGATAGTTGGGGGTATTCGTATTCAATTGTCAGAGGTGAAATTCTTGGATTTATGGAAGACGAACTACTGCGAAAGCATTTACCAAGGATGTTTTCATTA,ASV_14544,AWI,1.0,0.000000e+00,375.0,99.74,HQ867488.1.870_U|18S_rRNA|nucleus|clone_SHAH47...,379.0,HQ867488.1.870_U,18S_rRNA,...,clone_SHAH470,Eukaryota,TSAR,Stramenopiles,Gyrista,Chrysophyceae,Chrysophyceae_Clade-EC2H,Chrysophyceae_Clade-EC2H_X,Chrysophyceae_Clade-EC2H_XX,Chrysophyceae_Clade-EC2H_XX_sp.
TAGCGTATACTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGAATTTCTGGTCGCGAAAGGTGGTCCGTTCCTCACGGAATGTGTACTGTCCATTCGTGTCCATTCTCGGAGTTAGTTTGTATTACCTTCATCGGTCGTGCATACAACCTTCGTCATTTACTGTGAGCAAAATAGAGTGTTCAAAGCAGGCTTATGCCGTTGAATATATTAGCATGGAATAATAAGATATGACTTTGGTC

### Merge these together

In [73]:
# limit?
cols = ['ASV', 'Analyzing_Institute','bitscore', '%ID', 'hit_def']
df = pr2_blast_results[cols]
cols2 = ['ASV', 'Analyzing_Institute','bitscore', '%ID', 'ASV_hit']
df1 = blast_results_byseq[cols2]
df1 = df1.rename(columns={'ASV_hit':'hit_def'})
# df = pd.concat([pr2_blast_results[cols],blast_results_byseq[cols2] ], axis=1)
df = pd.concat([df, df1], axis=0)

# get rid of self-hits
df = df.loc[df['ASV']!=df['hit_def']]
# save for network analysis
file = '/Users/kpitz/github/GLOMICON/intercomparison/Merged_Datasets/network/blastnall_PR2_mergedhits.csv'
df.to_csv(file)
network_all = df.copy()

# limit to %ID >95:
df = df.loc[df['%ID']>=95]
# save for network analysis
file = '/Users/kpitz/github/GLOMICON/intercomparison/Merged_Datasets/network/blastnall_PR2_mergedhits_95ID.csv'
df.to_csv(file)
network_95 = df.copy()
df

,ASV,Analyzing_Institute,bitscore,%ID,hit_def
sequence,,,,,
CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATTTCGGGTCGGGCCGAGCGGTCTGCCGATGGGTATGCACTGTTTGGCGCGGCCTTCTTTCCGGAGACCGCGGCTACTCTTAACTGAGCGGGCGTGGGAGACGGATCGTTTACTTTGAAAAAATCAGAGTGTTTCTAGCAGGCAGCTCGCTCTTGCATAGGTTAGCATGGGATAATTTAATAGGACTCTGGTGCTATTTTGTTGGTTTCGAACACCGGAGTAATGATTAAAAGGGGCAGTCAGGGGCACTCGTATTCCGTCGAGAGAGGTGAAATTCTCAGACCAATGGAAGACGAACCACTGC,ASV_1,UDAL,350.0,100.0,KC488454.1.1667_U|18S_rRNA|nucleus|clone_HL5aS...
GCACCTACCGATTGAATGGTCCGGTGAAGACTCGGGATTGTGGTCTGGCTCCTTCATTGGGGCCAGACCGTGAGAACTTGTCTGAACCTTATCATTTAGAGGAAGGTGAAGTCGTAACAAGGTTTCC,ASV_2,MBARI,127.0,100.0,AJ535170.1.1763_U|18S_rRNA|nucleus|clone_p928|...
GCACCTACCGATTGAATGGTCCGGTGAGGCCTCGGGATCGTGGCGAACTTTCTTCATTGGAGGTGAGCTGTGAGAACTTGTCCAAATCTTATCATTTAGAGGAAGGTGAAGTCGTAACAAGGTTTCC,ASV_3,MBARI,118.0,97.64,KJ961704.1.1788_U|18S_rRNA|nucleus|strain_52|E...
GCACCTACCGATTGAATGGTCCGGTGAAGCCTCGGGATTGTGGTTGGTTTCCTTTATTGGAATCTGACCACGAGAACCTGTCTAAACCTTATCATTTAGAGGAAGGTGAAGTCGTAACAAGGTTTCC,ASV_4,MBARI,127.0,100.0,GU373969.1.1789_U|18S_rRNA|nucleus|strain_CCMP...
GCTCCTACCGATTGAGTGATCCGGTGAATAATTCGGACTGCAGCAGTGTTCAGTTCCTGAACGTTGCAGTGGAAAGTTTAGTGAACCTTATCACTTAGAGGAAGGAGAAGTCGTAACAAGGTTTCC,ASV_5,MBARI,126.0,100.0,MN965778.1.1765_U|18S_rRNA|nucleus||Eukaryota|...
...,...,...,...,...,...
TAGCGTATATTAAAGTTGTTGCGGTTAAAAAGCTCGTAGTTGGATTTCTGCTAGGGACGACCGGTCCGCCCACTGGGTTGGTATCTGGTTCGGCCTTGGCATCTTCTTGGAAGCCCGTATGCCATTGACTTGGTGTGCGGAGTTTCAAGACCTTTACTTTGAGGAAATTAGAGTGTTTCAAGCAGGCTTTCGCCTTGAATACATTAGCATGGAATAATAAGATAGGATCTCTTCCCTACCTTGTTGGTTCGTGGAGAAGAGATAATGATTAATAGGGATAGTCGGGGGCATTCGTACTTAACTGTCAGAGGTGAAATTCTTGGATTTGTTAAAGACGGACTACTGCGAAAGCATTTGCCAAGGATGTTTTCATTG,ASV_14546,AWI,369,99.47,ASV_10975
TAGCGTATATTTAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATTTCGGTTGAGAACGGCCGGTCCGCCGTTTGGTGTGCACTGGCTGGTCTCAACTTCCTGTAGAGGACGCGCTCTGGGTTAACGCTCGGACGCGGAGTCTACGTGGTTACTTTGAAAAAATTAGAGTGTTCAAAGCGGGCTTACGCTTGAATATTTCAGCATGGAATAACACTATAGGACTCCTGTCCTATTTCGTTGGTCTCGGGACGGGAGTAATGATTAAGAGGAACAGTTGGGGGCATTCGTATTTCATTGTCAGAGGTGAAATTCTTGGATTTATGAAAGACGAACTTCTGCGAAAGCATTTGCCAAGGATGTTTTCATTA,ASV_14547,AWI,367,100.0,ASV_11992
TAGCGTATATTTAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATTTCGGTTGAGAACGGCCGGTCCGCCGTTTGGTGTGCACTGGCTGGTCTCAACTTCCTGTAGAGGACGCGCTCTGGGTTAACGCTCGGACGCGGAGTCTACGTGGTTACTTTGAAAAAATTAGAGTGTTCAAAGCGGGCTTACGCTTGAATATTTCAGCATGGAATAACACTATAGGACTCCTGTCCTATTTCGTTGGTCTCGGGACGGGAGTAATGATTAAGAGGAACAGTTGGGGGCATTCGTATTTCATTGTCAGAGGTGAAATTCTTGGATTTATGAAAGACGAACTTCTGCGAAAGCATTTGCCAAGGATGTTTTCATTA,ASV_14547,AWI,366,99.73,ASV_11315


### Narrow to just one set of samples


In [89]:
df = asv_all.copy()
df = df.T
df = pd.concat([meta_all, df], axis=1)
print(df['site'].unique())
df = df.loc[df['site']=='EVENMOCK']
cols = list(meta_all)
df = df.drop(cols, axis=1)
df = df.T
df['total_reads'] = df.sum(axis=1)
df= df.loc[df['total_reads']>0]
df = df[['total_reads']]
#asvs in evenmock:
asvs_keep = df.index.tolist()
# join with network file:
df = network_95.join(df, on='ASV')
df= df.loc[df['total_reads']>0]
# get rid of hits to sequences NOT in sample group.
df = df.loc[(df['hit_def'].isin(asvs_keep)==True)|(df['hit_def'].str.contains('\|'))]
# save for network analysis
file = '/Users/kpitz/github/GLOMICON/intercomparison/Merged_Datasets/network/blastnall_PR2_mergedhits_95ID_evenmock.csv'
df.to_csv(file)
same_hits = df.copy()
df

['Monterey Bay' 'Fram Straight' 'Bedford Basin' nan 'BLOOMMOCK' 'EVENMOCK'
 'Scripps Pier' 'Western Channel' 'Roscoff']


<>:18: SyntaxWarning: invalid escape sequence '\|'
<>:18: SyntaxWarning: invalid escape sequence '\|'
/var/folders/5v/h4s_0pz17yb6f6krsxkzr5dw000b65/T/ipykernel_45071/1002134931.py:18: SyntaxWarning: invalid escape sequence '\|'
  df = df.loc[(df['hit_def'].isin(asvs_keep)==True)|(df['hit_def'].str.contains('\|'))]


,ASV,Analyzing_Institute,bitscore,%ID,hit_def,total_reads
sequence,,,,,,
CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTCGGATTTCGGGTCGGGCCGAGCGGTCTGCCGATGGGTATGCACTGTTTGGCGCGGCCTTCTTTCCGGAGACCGCGGCTACTCTTAACTGAGCGGGCGTGGGAGACGGATCGTTTACTTTGAAAAAATCAGAGTGTTTCTAGCAGGCAGCTCGCTCTTGCATAGGTTAGCATGGGATAATTTAATAGGACTCTGGTGCTATTTTGTTGGTTTCGAACACCGGAGTAATGATTAAAAGGGGCAGTCAGGGGCACTCGTATTCCGTCGAGAGAGGTGAAATTCTCAGACCAATGGAAGACGAACCACTGC,ASV_1,UDAL,350.0,100.0,KC488454.1.1667_U|18S_rRNA|nucleus|clone_HL5aS...,7011.0
GCACCTACCGATTGAATGGTCCGGTGAAGACTCGGGATTGTGGTCTGGCTCCTTCATTGGGGCCAGACCGTGAGAACTTGTCTGAACCTTATCATTTAGAGGAAGGTGAAGTCGTAACAAGGTTTCC,ASV_2,MBARI,127.0,100.0,AJ535170.1.1763_U|18S_rRNA|nucleus|clone_p928|...,44726.0
GCACCTACCGATTGAATGGTCCGGTGAGGCCTCGGGATCGTGGCGAACTTTCTTCATTGGAGGTGAGCTGTGAGAACTTGTCCAAATCTTATCATTTAGAGGAAGGTGAAGTCGTAACAAGGTTTCC,ASV_3,MBARI,118.0,97.64,KJ961704.1.1788_U|18S_rRNA|nucleus|strain_52|E...,39284.0
GCACCTACCGATTGAATGGTCCGGTGAAGCCTCGGGATTGTGGTTGGTTTCCTTTATTGGAATCTGACCACGAGAACCTGTCTAAACCTTATCATTTAGAGGAAGGTGAAGTCGTAACAAGGTTTCC,ASV_4,MBARI,127.0,100.0,GU373969.1.1789_U|18S_rRNA|nucleus|strain_CCMP...,101282.0
GCTCCTACCGATTGAGTGATCCGGTGAATAATTCGGACTGCAGCAGTGTTCAGTTCCTGAACGTTGCAGTGGAAAGTTTAGTGAACCTTATCACTTAGAGGAAGGAGAAGTCGTAACAAGGTTTCC,ASV_5,MBARI,126.0,100.0,MN965778.1.1765_U|18S_rRNA|nucleus||Eukaryota|...,87.0
...,...,...,...,...,...,...
TAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATTTCTGGCAGGAGCGACCGGTCACGCACTCTGTGCGTGAACTTGTGTTGTCTCTGGCCATCCTTGGGGAGATCCTGTTTGGCATTAAGTTGTCGGGCAGGGGATACCCATCGTTTACTGTGAAAAAATTAGAGTGTTTAAAGCAGGCTTATGCCGTTGAATATATTAGCATGGAATAATAAGATAGGACTTCGGAACTATTTTGTTGGTTTGCGTTACGAAGTAATGATTAATAGGGACAGTTGGGGGTATTCGTATTGCATTGTCAGAGGTGAAATTCTTGGATTTCTGCAAGACGAACTACTGCGAAAGCATTTACCAAGGATGTTTTCATTA,ASV_14327,AWI,376,99.74,ASV_11587,11.0
TAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATTTCTGGCAGGAGCGACCGGTCACGCACTCTGTGCGTGAACTTGTGTTGTCTCTGGCCATCCTTGGGGAGATCCTGTTTGGCATTAAGTTGTCGGGCAGGGGATACCCATCGTTTACTGTGAAAAAATTAGAGTGTTTAAAGCAGGCTTATGCCGTTGAATATATTAGCATGGAATAATAAGATAGGACTTCGGAACTATTTTGTTGGTTTGCGTTACGAAGTAATGATTAATAGGGACAGTTGGGGGTATTCGTATTGCATTGTCAGAGGTGAAATTCTTGGATTTCTGCAAGACGAACTACTGCGAAAGCATTTACCAAGGATGTTTTCATTA,ASV_14327,AWI,373,99.47,ASV_11383,11.0
TAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATTTCTGGCAGGAGCGACCGGTCACGCACTCTGTGCGTGAACTTGTGTTGTCTCTGGCCATCCTTGGGGAGATCCTGTTTGGCATTAAGTTGTCGGGCAGGGGATACCCATCGTTTACTGTGAAAAAATTAGAGTGTTTAAAGCAGGCTTATGCCGTTGAATATATTAGCATGGAATAATAAGATAGGACTTCGGAACTATTTTGTTGGTTTGCGTTACGAAGTAATGATTAATAGGGACAGTTGGGGGTATTCGTATTGCATTGTCAGAGGTGAAATTCTTGGATTTCTGCAAGACGAACTACTGCGAAAGCATTTACCAAGGATGTTTTCATTA,ASV_14327,AWI,373,99.47,ASV_10416,11.0


In [107]:
### Manually look at groups that hit same PR2 sequence across all institutes:
common_hits = []

df = same_hits.copy()
df = df.reset_index()
# just PR2 results
df = df.loc[(df['hit_def'].str.contains('\|'))]
# df = df.groupby(['hit_def', 'Analyzing_Institute', 'ASV']).sum()
for i, group in df.groupby('hit_def'):
    if len(group['Analyzing_Institute'].unique()) == 5:
        print(i)
        #print(group)
        common_hits.append(i)
df = df.loc[df['hit_def'].isin(common_hits)]
#df = df.groupby(['hit_def', 'Analyzing_Institute', 'ASV']).sum()
df = df.loc[df['hit_def'].isin(['AB430605.1.1762_U|18S_rRNA|nucleus|strain_s0388|Eukaryota|TSAR|Stramenopiles|Gyrista|Bacillariophyceae|Fragilariales|Staurosiraceae|Plagiostriata|Plagiostriata_goreensis'])]
# export fasta file:

df['label'] = df['ASV'] +'|' + df['Analyzing_Institute']+ '|' + df['total_reads'].astype(str) + '|' + df['%ID'].astype(str)

outfile = '/Users/kpitz/github/GLOMICON/intercomparison/Merged_Datasets/network/'+'Plagiostriata_goreensis.fasta'

seqs = df['sequence'].tolist()
otus = df.label.tolist()

##Write File
resultsFile = open(outfile, "w") #open resultsfile

for i in range(len(otus)):
    resultsFile.write('>' + otus[i])
    resultsFile.write('\n')
    resultsFile.write(seqs[i])
    resultsFile.write('\n')       
print("End")
resultsFile.close()




# print(list(df['ASV'].unique()))
# print(list(df['sequence']))
df

AB430605.1.1762_U|18S_rRNA|nucleus|strain_s0388|Eukaryota|TSAR|Stramenopiles|Gyrista|Bacillariophyceae|Fragilariales|Staurosiraceae|Plagiostriata|Plagiostriata_goreensis
AJ535170.1.1763_U|18S_rRNA|nucleus|clone_p928|Eukaryota|TSAR|Stramenopiles|Gyrista|Mediophyceae|Thalassiosirales|Thalassiosiraceae|Conticribra|Conticribra_weissflogii
EU371222.1.1803_U|18S_rRNA|nucleus|clone_NPK55_1|Eukaryota|TSAR|Stramenopiles|Gyrista|Mediophyceae|Thalassiosirales|Thalassiosiraceae|Thalassiosira|Thalassiosira_hispida
End


<>:7: SyntaxWarning: invalid escape sequence '\|'
<>:7: SyntaxWarning: invalid escape sequence '\|'
/var/folders/5v/h4s_0pz17yb6f6krsxkzr5dw000b65/T/ipykernel_45071/541902277.py:7: SyntaxWarning: invalid escape sequence '\|'
  df = df.loc[(df['hit_def'].str.contains('\|'))]


,sequence,ASV,Analyzing_Institute,bitscore,%ID,hit_def,total_reads,label
82,AGCTCCAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTA...,ASV_116,SBR,365.0,98.44,AB430605.1.1762_U|18S_rRNA|nucleus|strain_s038...,4538.0,ASV_116|SBR|4538.0|98.44
115,CAATAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGG...,ASV_207,UDAL,332.0,98.29,AB430605.1.1762_U|18S_rRNA|nucleus|strain_s038...,1827.0,ASV_207|UDAL|1827.0|98.29
146,GCACCTACCGATTGAATGGTCCGGTGAAGCCTCGGGATCGCGGCGA...,ASV_305,MBARI,118.0,97.64,AB430605.1.1762_U|18S_rRNA|nucleus|strain_s038...,1096.0,ASV_305|MBARI|1096.0|97.64
178,GCACCTACCGATTGAATGGTCCGGTGAAGCCTCGGGATCGCGGCGA...,ASV_455,NOAA,118.0,97.64,AB430605.1.1762_U|18S_rRNA|nucleus|strain_s038...,566.0,ASV_455|NOAA|566.0|97.64
697,TAGCGTATATTAAAGTTGTTGCAGTTAAAAAGCTCGTAGTTGGATT...,ASV_11683,AWI,360.0,98.42,AB430605.1.1762_U|18S_rRNA|nucleus|strain_s038...,22244.0,ASV_11683|AWI|22244.0|98.42
